In [44]:
import urllib.request, zipfile, os
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# 1. **Load the Data**:

Using the MovieLens 100K Dataset.
i) Data sample is already clean, so we won't need to account for null values
ii) contains 1-5 star ratings.


In [45]:
if not os.path.exists('ml-100k/u.data'):
    urllib.request.urlretrieve('https://files.grouplens.org/datasets/movielens/ml-100k.zip', 'ml-100k.zip')
    zipfile.ZipFile('ml-100k.zip').extract('ml-100k/u.data')

df = pd.read_csv('ml-100k/u.data', sep='\t',         #MovieLens 100K is a tab separated file
                      names=['user_id', 'movie_id', 'rating', 'timestamp'],       #the original u.data file doesnt have a header at top. so this gives them header names
)

In [46]:
print(df.shape)
df.head()

(100000, 4)


,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [47]:
df.info()
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   user_id    100000 non-null  int64
 1   movie_id   100000 non-null  int64
 2   rating     100000 non-null  int64
 3   timestamp  100000 non-null  int64
dtypes: int64(4)
memory usage: 3.1 MB


,0
user_id,0
movie_id,0
rating,0
timestamp,0


# 2. **Train Test Split**:

Split the data in 80-20 before performing EDA, to prevent data-leakage.
We split the data such that we use a Temporal Split- train on older ratings, test on newer ratings -> We require the timestamps to be arranged in ascending order

In [48]:
df = df.sort_values('timestamp').reset_index(drop=True)

In [49]:
split_idx = int(len(df) * 0.8)

train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

In [50]:
#Dropping timestamp column
train_df = train_df.drop(columns=['timestamp'])
test_df = test_df.drop(columns=['timestamp'])

print(f"Train size: {len(train_df)} rows | Test size: {len(test_df)} rows")

Train size: 80000 rows | Test size: 20000 rows


Cons of a time-based (Temporal) Split of train-test: If a user only rated movies during the final 20% of the timeline, they will only be in the test set, creating a "cold start" user problem for matrix factorization